# Lost in the Museum - rotation-invariant DINOv2 ViT-g/14 @ 518px

Same backbone and resolution as the run that scored **0.8285**, with each image
described by the average of its embeddings at 0, 90, 180 and 270 degrees.

### Why

Inspecting what the model actually retrieved for real queries turned up the
cause. Query `02200` was, by eye, its own top-1 match rotated 180 degrees --
the same two children, the same red bow, upside down. DINOv2 has no invariance
to that, so it scored the pair at 0.280 and nearly lost it.

A survey of 45 detected queries found this is common, not incidental:

| Best-matching orientation | Share |
|---|---|
| 0 deg (upright) | 56% |
| 90 deg | 4% |
| **180 deg** | **29%** |
| 270 deg | 11% |

**44% of queries are rotated by a multiple of 90 degrees.** No amount of better
features helps there, because the model is being shown a picture it has no way
to relate to the upright original. On a confirmed pair, averaging over the four
orientations lifted cosine similarity from **0.790 to 0.981**.

It also explains the leaderboard: handling orientation is a binary capability,
so everyone who solves it lands near 0.96 while everyone who does not sits
around 0.83.

### Why averaging rather than detecting

Averaging maps every rotation of an image to the same descriptor, so an
upside-down query and its upright gallery original agree *by construction*.
Detecting the "correct" orientation would preserve more detail, but a wrong
guess is unrecoverable. Averaging cannot fail catastrophically -- both sides
receive identical treatment.

Cost is 4x the forward passes. **Expect roughly 5-7 hours**; the session limit
is 12. Checkpointing is included so an interruption resumes.


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None


def find_data_dir():
    best, best_n = None, 0
    for d in Path("/kaggle/input").rglob("*"):
        if d.is_dir():
            n = sum(1 for _ in d.glob("*.png"))
            if n > best_n:
                best, best_n = d, n
    return best, best_n


DATA_DIR, _n = find_data_dir()
print("Data:", DATA_DIR, f"({_n} png)")
assert DATA_DIR is not None and _n == 20000

MODEL, SIZE, DIM, BATCH = "dinov2_vitg14", 518, 1536, 4
ROTATIONS = (0, 90, 180, 270)
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
WORK = Path("/kaggle/working")

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Enable Settings -> Accelerator -> GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}  "
      f"VRAM {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")

## Dataset

Each item yields the four rotations of one image as a single stacked tensor, so
the orientations stay together in a batch and the averaging is exact.


In [ ]:
class RotationSet(Dataset):
    """Returns a (4, 3, H, W) tensor: one image at every 90-degree rotation."""

    def __init__(self, paths, size):
        self.paths, self.size = paths, size
        self.tf = transforms.Compose([
            transforms.Resize((size, size),
                              interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
            views = [self.tf(img.rotate(a, expand=True)) for a in ROTATIONS]
            return torch.stack(views), i
        except Exception as e:
            print(f"  ! failed {self.paths[i].name}: {e}")
            # never drop a row -- the submission must carry all 20,000 images
            return torch.zeros(len(ROTATIONS), 3, self.size, self.size), i


def gem_pool(patch_tokens, p=3.0, eps=1e-6):
    return patch_tokens.clamp(min=eps).pow(p).mean(dim=1).pow(1.0 / p)

## Embed

Each orientation is L2-normalised *before* averaging, so no single view
dominates by magnitude; the mean is then renormalised.


In [ ]:
paths = sorted(DATA_DIR.glob("*.png"))
model = torch.hub.load("facebookresearch/dinov2", MODEL, verbose=False).eval().to(device).half()

CK, CKD = WORK / "rot_ckpt.npy", WORK / "rot_done.npy"
feats, done = None, np.zeros(len(paths), dtype=bool)
if CK.exists() and CKD.exists():
    feats, done = np.load(CK), np.load(CKD)
    print(f"resuming: {done.sum()}/{len(paths)} already embedded")

todo = np.flatnonzero(~done)
loader = DataLoader(torch.utils.data.Subset(RotationSet(paths, SIZE), todo.tolist()),
                    batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)

t0 = last = time.time()
with torch.no_grad():
    for views, idxs in loader:
        b, r, c, h, w = views.shape
        flat = views.view(b * r, c, h, w).to(device, non_blocking=True).half()
        out = model.forward_features(flat)
        v = torch.cat([
            F.normalize(out["x_norm_clstoken"], dim=1),
            F.normalize(gem_pool(out["x_norm_patchtokens"]), dim=1),
        ], dim=1)
        v = F.normalize(v.view(b, r, -1).mean(dim=1), dim=1).float().cpu().numpy()

        if feats is None:
            feats = np.zeros((len(paths), v.shape[1]), dtype=np.float32)
        feats[idxs.numpy()] = v
        done[idxs.numpy()] = True

        if time.time() - last > 300:
            np.save(CK, feats); np.save(CKD, done); last = time.time()
        n = int(done.sum())
        if n % (BATCH * 100) < BATCH:
            rate = (n - (len(paths) - len(todo))) / (time.time() - t0)
            print(f"  {n}/{len(paths)}  {rate:.1f} img/s  "
                  f"ETA {(len(paths)-n)/max(rate,1e-6)/60:.0f} min", flush=True)

assert done.all(), f"only {done.sum()}/{len(paths)} embedded"
print(f"embedded {feats.shape} in {(time.time()-t0)/60:.1f} min")
np.save(WORK / "features_rot.npy", feats)
for f in (CK, CKD):
    f.unlink(missing_ok=True)

## Whitened PCA and submission


In [ ]:
def l2(x, eps=1e-12):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)


x = l2(feats.astype(np.float64))
mu = x.mean(axis=0, keepdims=True)
_, s, vt = np.linalg.svd(x - mu, full_matrices=False)
dim = min(DIM, x.shape[1])
x = l2((x - mu) @ vt[:dim].T / (s[:dim] / np.sqrt(len(x) - 1) + 1e-8)).astype(np.float32)
print(f"PCA {feats.shape[1]} -> {dim}  explains {(s[:dim]**2).sum()/(s**2).sum():.1%}")

df = pd.DataFrame(x, columns=[f"feature_{i}" for i in range(x.shape[1])])
df.insert(0, "image_name", [p.name for p in paths])
df["ID"] = df["image_name"]
df.to_csv(WORK / "submission.csv", index=False, float_format="%.6f")
print(f"rows={len(df)}  cols={df.shape[1]}")
df.iloc[:3, :5]